In [1]:
# import os
# current_path = os.getcwd()
# print(current_path)

/Users/sofianekrasova/testov/qa


In [2]:
# import sys
# print(sys.executable)

/Users/sofianekrasova/Library/jupyterlab-desktop/jlab_server/bin/python


Важный момент!! 

после скачивания файлов важно открыть скрытый файлик .env в корне папки и проверить ключ api openroute. сейчас там стоит мой ключ какой-то бесплатный, потом можно поменять. не знаю, какой там лимит

настраиваем окружение и импорты + пути к файлам нас интересующих

In [3]:
import os
import sys
import json
import logging
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

# Находим корневую папку проекта 
notebook_path = os.path.abspath('.')
project_root = os.path.dirname(notebook_path) 
sys.path.insert(0, project_root)

# Загружаем переменные из .env файла, который лежит в корне проекта
env_path = os.path.join(project_root, '.env')
if os.path.exists(env_path):
    load_dotenv(env_path)
    print(f"Переменные окружения загружены из: {env_path}")
else:
    print("Файл .env не найден! Проверь, что он лежит в корневой папке проекта.")

# Настраиваем логирование, чтобы видеть прогресс
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Импорт компонентов системы
try:
    from app.orchestrator.runner import run
    from app.judge.agent import JudgeAgent
    from app.models.state import GraphState
    from app.models.schemas import Verdict, IssueType
    from langchain_core.messages import HumanMessage
    from langchain_openai import ChatOpenAI
    from app.config import get_config
    print("Все модули успешно импортированы")
except ImportError as e:
    logger.error(f"Ошибка импорта: {e}. Убедись, что ты установил зависимости через pip3.11 install -r requirements.txt")

# Пути к датасетам
DATA_DIR = os.path.join(project_root, 'data')
SAFE_DATASET_PATH = os.path.join(DATA_DIR, 'dataset.csv')
VULN_DATASET_PATH = os.path.join(DATA_DIR, 'vuln_dataset.csv')

Переменные окружения загружены из: /Users/sofianekrasova/testov/.env
Все модули успешно импортированы


функции загрузки данных и метрик 

In [4]:
def load_safe_dataset():
    if not os.path.exists(SAFE_DATASET_PATH):
        logger.error(f"Файл {SAFE_DATASET_PATH} не найден!")
        return []
    try:
        df = pd.read_csv(SAFE_DATASET_PATH, sep=';', on_bad_lines='skip')
        # Проверяем наличие нужных колонок, если их нет, пробуем стандартные имена
        if 'text' not in df.columns and 'task_description' in df.columns:
             df.rename(columns={'task_description': 'text'}, inplace=True)
        return df.to_dict('records')
    except Exception as e:
        logger.error(f"Ошибка чтения safe dataset: {e}")
        return []

def load_vulnerable_dataset():
    if not os.path.exists(VULN_DATASET_PATH):
        logger.error(f"Файл {VULN_DATASET_PATH} не найден!")
        return []
    try:
        df = pd.read_csv(VULN_DATASET_PATH, sep=',', on_bad_lines='skip')
        return df.to_dict('records')
    except Exception as e:
        logger.error(f"Ошибка чтения vuln dataset: {e}")
        return []

def calculate_execution_accuracy(results):
    if not results: return 0.0
    approved_count = sum(1 for r in results if r.get('approved', False))
    return round((approved_count / len(results)) * 100, 2)

def calculate_judge_coverage(judge_results):
    total_tests = len(judge_results)
    detected_correctly = 0
    missed_vulns = []
    found_vuln_types = set()

    for res in judge_results:
        expected_vuln_class = res.get('expected_vuln_class')
        found_issues = res.get('found_issues', [])
        
        has_critical_issue = any(i.severity.value in ['HIGH', 'MEDIUM'] for i in found_issues)
        is_rejected = res.get('verdict') == Verdict.REJECTED
        
        for issue in found_issues:
            found_vuln_types.add(issue.type.value)

        if has_critical_issue or is_rejected:
            detected_correctly += 1
        else:
            missed_vulns.append({
                "id": res.get('id'),
                "expected_vuln": expected_vuln_class,
                "verdict": res.get('verdict'),
                "issues_found": [i.type.value for i in found_issues]
            })
            
    coverage = (detected_correctly / total_tests) * 100 if total_tests > 0 else 0
    
    return {
        "coverage_percent": round(coverage, 2),
        "total_tests": total_tests,
        "detected": detected_correctly,
        "unique_vuln_types_found": list(found_vuln_types),
        "missed_cases": missed_vulns
    }

def analyze_iterations(pipeline_results):
    if not pipeline_results: return {}
    iterations_list = [r['iterations_used'] for r in pipeline_results]
    avg_iterations = sum(iterations_list) / len(iterations_list)
    
    risk_reduction_success = 0
    total_risk_start = 0
    total_risk_end = 0

    for r in pipeline_results:
        logs = r.get('iteration_logs', [])
        if logs:
            initial_risk = logs[0].judge_output.risk_score
            final_risk = logs[-1].judge_output.risk_score
            total_risk_start += initial_risk
            total_risk_end += final_risk
            if final_risk < initial_risk or (final_risk == 0 and r['approved']):
                risk_reduction_success += 1
                
    reduction_rate = (risk_reduction_success / len(pipeline_results)) * 100 if pipeline_results else 0
    avg_risk_start = total_risk_start / len(pipeline_results)
    avg_risk_end = total_risk_end / len(pipeline_results)

    return {
        "avg_iterations": round(avg_iterations, 2),
        "max_iterations": max(iterations_list) if iterations_list else 0,
        "min_iterations": min(iterations_list) if iterations_list else 0,
        "risk_reduction_rate": round(reduction_rate, 2),
        "avg_risk_start": round(avg_risk_start, 2),
        "avg_risk_end": round(avg_risk_end, 2)
    }

функции тестирования пайплайна и судьи

In [5]:
def test_full_pipeline(safe_tasks):
    logger.info("Запуск теста полного пайплайна (Safe Tasks)")
    results = []
    # Берем первые 50 задач для скорости, или все если нужно
    tasks_to_test = safe_tasks[:50] 
    
    for task_data in tasks_to_test:
        task_desc = task_data.get('text', '') or task_data.get('task_description', '')
        if not task_desc or len(str(task_desc)) < 5:
            continue

        logger.info(f"Обработка задачи: {str(task_desc)[:40]}...")
        try:
            # Вызываем основной раннер системы
            result = run(str(task_desc))
            results.append({
                "task_preview": str(task_desc)[:50],
                "final_sql": result.final_sql,
                "approved": result.approved,
                "iterations": result.iterations_used,
                "logs": result.iteration_logs
            })
        except Exception as e:
            logger.error(f"Ошибка при выполнении задачи: {e}")
            results.append({
                "task_preview": str(task_desc)[:50],
                "error": str(e),
                "approved": False,
                "iterations": 0
            })
    return results

def test_judge_agent(vuln_tasks):
    logger.info("=== Запуск теста агента-судьи (Vuln Tasks) ===")
    results = []
    
    # Получаем конфигурацию для создания LLM
    config = get_config()
    
    # Важно: проверяем, что модель указана в конфиге/переменных окружения
    if not config.model_name:
        raise ValueError("MODEL_NAME не найдена в .env файле или config.py")

    llm = ChatOpenAI(
        base_url=config.openrouter_base_url,
        api_key=config.open_router_api_key,
        model=config.model_name
    )
    
    judge = JudgeAgent(llm=llm)
    
    for vuln_data in vuln_tasks:
        vuln_sql = vuln_data.get('vulnerable_sql', '')
        expected_vuln = vuln_data.get('expected_vuln_class', '')
        
        if not vuln_sql:
            continue

        logger.info(f"Проверка уязвимости [{expected_vuln}]: {vuln_sql[:30]}...")
        
        state = {
            "messages": [HumanMessage(content=f"Проверь этот SQL на безопасность: {vuln_sql}")],
            "judge_responses": [],
            "llm_calls": 0,
            "audit_iteration": 0
        }
        
        try:
            output = judge(state)
            last_response = output['judge_responses'][-1]
            
            results.append({
                "id": vuln_data.get('id'),
                "sql_preview": vuln_sql[:50],
                "expected_vuln_class": expected_vuln,
                "verdict": last_response.verdict,
                "risk_score": last_response.risk_score,
                "found_issues": last_response.issues
            })
        except Exception as e:
            logger.error(f"Ошибка судьи на задаче: {e}")
            
    return results

а теперь главный запуск и вывод результатов

In [6]:
print("Начало комплексного тестирования системы secure-sql-agents")
print("-" * 60)

# 1. Загрузка данных
safe_tasks = load_safe_dataset()
vuln_tasks = load_vulnerable_dataset()

if not safe_tasks and not vuln_tasks:
    print("Ошибка: Датасеты не загружены. Проверь пути к файлам в папке data/")
else:
    # 2. Тестирование
    pipeline_results = []
    judge_results = []

    if safe_tasks:
        pipeline_results = test_full_pipeline(safe_tasks)
    
    if vuln_tasks:
        judge_results = test_judge_agent(vuln_tasks)

    # 3. Расчет метрик
    exec_accuracy = calculate_execution_accuracy(pipeline_results)
    judge_stats = calculate_judge_coverage(judge_results)
    iter_stats = analyze_iterations(pipeline_results)

    # 4. Красивый вывод в Notebook
    print("\n" + "=" * 60)
    print("ИТОГОВЫЙ ОТЧЕТ ПО ТЕСТИРОВАНИЮ")
    print("=" * 60)
    
    if pipeline_results:
        print(f"\n1. Точность генерации (Execution Accuracy): **{exec_accuracy}%**")
        status_acc = "Пройдено" if exec_accuracy >= 70 else "Не пройдено"
        print(f"   Статус: {status_acc} (Цель МФТИ: >= 70%)")
        
        print(f"\n2. Эффективность итераций:")
        print(f"   Среднее число итераций: {iter_stats['avg_iterations']}")
        print(f"   Снижение риска: {iter_stats['avg_risk_start']} -> {iter_stats['avg_risk_end']}")
        print(f"   Доля случаев с исправлением: {iter_stats['risk_reduction_rate']}%")

    if judge_results:
        print(f"\n3. Покрытие уязвимостей судьей: **{judge_stats['coverage_percent']}%**")
        print(f"   Найдено корректно: {judge_stats['detected']} из {judge_stats['total_tests']}")
        print(f"   Уникальных типов угроз обнаружено: {len(judge_stats['unique_vuln_types_found'])}")
        print(f"   Типы: {', '.join(judge_stats['unique_vuln_types_found'])}")
        
        if judge_stats['missed_cases']:
            print("   ⚠️ Пропущенные случаи (примеры):")
            for case in judge_stats['missed_cases'][:3]:
                print(f"     - ID {case['id']}: ожидалось {case['expected_vuln']}, вердикт {case['verdict']}")
        else:
            print("   Все уязвимости обнаружены успешно")

    # 5. Сохранение отчета
    report = {
        "timestamp": datetime.now().isoformat(),
        "metrics": {
            "execution_accuracy": exec_accuracy,
            "judge_coverage": judge_stats['coverage_percent'],
            "avg_iterations": iter_stats.get('avg_iterations', 0),
            "risk_reduction_rate": iter_stats.get('risk_reduction_rate', 0)
        },
        "pipeline_details": pipeline_results,
        "judge_details": judge_results
    }
    
    reports_dir = os.path.join(project_root, 'qa', 'reports')
    os.makedirs(reports_dir, exist_ok=True)
    report_filename = os.path.join(reports_dir, f"test_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")
    
    with open(report_filename, 'w', encoding='utf-8') as f:
        json.dump(report, f, indent=2, ensure_ascii=False, default=str)
        
    print(f"\n Подробный отчет сохранен в: `{report_filename}`")
    print("-" * 60)

2026-05-15 20:22:20,615 - ERROR - Ошибка чтения safe dataset: Error tokenizing data. C error: EOF inside string starting at row 1759
2026-05-15 20:22:20,627 - INFO - === Запуск теста агента-судьи (Vuln Tasks) ===


🚀 Начало комплексного тестирования системы secure-sql-agents
------------------------------------------------------------


2026-05-15 20:22:22,655 - INFO - Проверка уязвимости [SQL_INJ_CLASSIC]: SELECT * FROM logs WHERE msg =...
2026-05-15 20:22:25,425 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-15 20:22:26,629 - INFO - Проверка уязвимости [SQL_INJ_CLASSIC]: SELECT * FROM orders WHERE sta...
2026-05-15 20:22:27,205 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-15 20:22:27,830 - INFO - Проверка уязвимости [SQL_INJ_CLASSIC]: EXECUTE 'SELECT * FROM users W...
2026-05-15 20:22:28,338 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-15 20:22:29,276 - INFO - Проверка уязвимости [SQL_INJ_CLASSIC]: SELECT * FROM products WHERE n...
2026-05-15 20:22:29,773 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-15 20:22:30,750 - INFO - Проверка уязвимости [SQL_INJ_CLASSIC]: SELECT * FROM customer WHERE l...
2026-0


📊 ИТОГОВЫЙ ОТЧЕТ ПО ТЕСТИРОВАНИЮ

3. Покрытие уязвимостей судьей: **94.44%**
   Найдено корректно: 51 из 54
   Уникальных типов угроз обнаружено: 2
   Типы: SECURITY, PERFORMANCE
   ⚠️ Пропущенные случаи (примеры):
     - ID 44: ожидалось NO_PAGINATION, вердикт Verdict.APPROVED
     - ID 45: ожидалось NO_PAGINATION, вердикт Verdict.APPROVED
     - ID 47: ожидалось NO_PAGINATION, вердикт Verdict.APPROVED

💾 Подробный отчет сохранен в: `/Users/sofianekrasova/testov/qa/reports/test_report_20260515_202345.json`
------------------------------------------------------------
